# Industrial Anomaly Benchmark Colab

In [ ]:
import sys, platform
print(sys.version)
print(platform.platform())
!nvidia-smi || true

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/OWNER/REPO.git'  # set before running
BRANCH = 'main'
DRIVE_DATA_ROOT = '/content/drive/MyDrive/industrial-anomaly-benchmark-data'
DRIVE_RESULT_ROOT = '/content/drive/MyDrive/industrial-anomaly-benchmark-results'
REPO_DIR = '/content/INFORMS_FORD'

In [ ]:
from pathlib import Path
if Path(REPO_DIR).exists():
    %cd $REPO_DIR
    !git fetch origin && git checkout $BRANCH && git pull --ff-only
else:
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR

In [ ]:
!pip install -e .

In [ ]:
ACCEPT_DIR = f'{DRIVE_DATA_ROOT}/ford/Train/accept'
REJECT_DIR = f'{DRIVE_DATA_ROOT}/ford/Train/reject'
!python scripts/validate_dataset.py --config configs/datasets/ford.yaml --accepted-dir "$ACCEPT_DIR" --rejected-dir "$REJECT_DIR"

In [ ]:
RAW_DIR = f'{DRIVE_RESULT_ROOT}/raw'
!python scripts/run_experiment.py --config configs/experiments/ford_isolation_forest.yaml --accepted-dir "$ACCEPT_DIR" --rejected-dir "$REJECT_DIR" --output-dir "$RAW_DIR"

In [ ]:
AGG_DIR = f'{DRIVE_RESULT_ROOT}/aggregated'
!python scripts/build_results_table.py --results-dir "$RAW_DIR" --output-dir "$AGG_DIR"

In [ ]:
import pandas as pd
pd.read_csv(f'{AGG_DIR}/benchmark_results.csv')